# Notebook 02 — D2–D5 source provenance and schema audit

**Run All, CPU only.** One executable cell audits **four candidate datasets**: D2 HarmJ0y (5 source files), D3 PhishFuzzer (2 files), D4 LLMGen (2 files), D5 Gutierrez origin-attribution archive (1 ZIP). Sources are pinned to published GitHub revisions, a Hugging Face revision or the Zenodo MD5. In each case the notebook records downloaded SHA256, sizes, schema, class metadata, duplicate counts and parent-link coverage **without printing email text, PII, or live phishing links**. An offline synthetic self-test runs first.

The notebook first records Python, OS, packages, CPU cores, available RAM/disk, optional GPU information and measured runtime/peak RSS. Public aggregate reports are saved under `/kaggle/working/phishing_d2_d5_evidence/`. Temporary third-party raw datasets are held under `/kaggle/temp/phishing_d2_d5_private/` (on Kaggle) and are **not** shareable outputs. Attached Kaggle Inputs are considered only where they can be checked against a pinned Git blob or published MD5; no stored source scripts are executed.

**Scientific scope:** D2–D5 data intake only, not training. Source labels may mean phishing, spam, legitimate or AI/human authorship, and original↔rewrite parent links, rights, label truth and upstream source overlap remain UNVERIFIED until later audit and human review. D5 is authorship-only, not the binary phishing task. Cross-file text overlap here is descriptive, not proof of independent experimental splits.

**Inputs / network:** Kaggle Internet ON for downloading missing sources. If offline, attach independently obtained source files matching pinned identities where supported. Download budget approximately 160 MB. No GPU necessary.

In [ ]:
# NOTEBOOK 02 — complete CPU-only D2–D5 intake; safe aggregates, no data execution
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations
import csv, hashlib, importlib.metadata, io, json, os, platform, re
import shutil, subprocess, sys, tempfile, time, urllib.parse, urllib.request, zipfile

RELEASE = "D2D5-N02-v1.0-2026-10-31"
REV_D2 = "d13e9bb73d03f458e6d5b6cb6c7b0a708ea47e3d"
REV_D3 = "1e21dd4edbe5c64694f156bf5318c97c7c80681c"
REV_D4 = "d717e54"  # HF repository commit displayed on source tree; no branch fallback
ON_KAGGLE = Path("/kaggle").is_dir()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
TEMP = (Path("/kaggle/temp") if ON_KAGGLE else Path(tempfile.gettempdir())) / "phishing_d2_d5_private"
OUT = ROOT / "phishing_d2_d5_evidence"
TEMP.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

# File sizes and blob IDs are from the two pinned public GitHub trees.
SOURCES = [
 {"id":"D2_original", "family":"D2", "kind":"github", "repo":"HarmJ0y/PhishingSpamDataSet",
  "rev":REV_D2,"path":"1_DataSet/cleaned_emails.json","name":"cleaned_emails.json",
  "size":7370034,"blob":"b9d6dd2ac3d6e2d337b9b9d79fc8f1f4377ea4f9"},
 {"id":"D2_rephrase_deepseek", "family":"D2", "kind":"github", "repo":"HarmJ0y/PhishingSpamDataSet",
  "rev":REV_D2,"path":"3_RePhrase/DeepSeek/emails_rephrased_clean_deepseek.json","name":"emails_rephrased_clean_deepseek.json",
  "size":7528880,"blob":"11cc02aeb03b5a318f3107a71a20f2406b8f790a"},
 {"id":"D2_rephrase_gpt4o", "family":"D2", "kind":"github", "repo":"HarmJ0y/PhishingSpamDataSet",
  "rev":REV_D2,"path":"3_RePhrase/GPT4/emails_rephrased_clean_openai4o.json","name":"emails_rephrased_clean_openai4o.json",
  "size":7079916,"blob":"d0cf5fe3bdf0a3dd962492720c3cb10381f9e313"},
 {"id":"D2_rephrase_randomapi", "family":"D2", "kind":"github", "repo":"HarmJ0y/PhishingSpamDataSet",
  "rev":REV_D2,"path":"3_RePhrase/RandomAPI/emails_rephrased_openrouter_clean.json","name":"emails_rephrased_openrouter_clean.json",
  "size":7124691,"blob":"2febaf0e74ac23689889164bfe5b8e538b706140"},
 {"id":"D2_categorized", "family":"D2", "kind":"github", "repo":"HarmJ0y/PhishingSpamDataSet",
  "rev":REV_D2,"path":"5_Categorize/merged_emails_with_categories.jsonl",
  "name":"merged_emails_with_categories.jsonl","size":31826514,
  "blob":"8c6d8ab36dfe2c16b441a88288d44790372aa68d"},
 {"id":"D3_seeds", "family":"D3", "kind":"github", "repo":"DataPhish/PhishFuzzer",
  "rev":REV_D3,"path":"PhishFuzzer_emails_original_seed_v1.json",
  "name":"PhishFuzzer_emails_original_seed_v1.json","size":8218997,
  "blob":"294e84ed06957ab64b3a6fe4dfc05a6afda7f7de"},
 {"id":"D3_variants", "family":"D3", "kind":"github", "repo":"DataPhish/PhishFuzzer",
  "rev":REV_D3,"path":"PhishFuzzer_emails_entity_rephrased_v1.json",
  "name":"PhishFuzzer_emails_entity_rephrased_v1.json","size":48159145,
  "blob":"b498a70203160940291156e07f10bcbb1edb9755"},
 {"id":"D4_chinese", "family":"D4", "kind":"hf", "rev":REV_D4,
  "path":"CN_Phishing_Email_dataset.csv","name":"CN_Phishing_Email_dataset.csv","size_cap":7_000_000},
 {"id":"D4_english", "family":"D4", "kind":"hf", "rev":REV_D4,
  "path":"GPT_Phishing_Email_dataset.csv","name":"GPT_Phishing_Email_dataset.csv","size_cap":9_000_000},
 {"id":"D5_archive", "family":"D5", "kind":"zenodo",
  "path":"cross-model-phishing.zip","name":"cross-model-phishing.zip","size_cap":15_000_000,
  "md5":"7bd6097768e0f44e433a153a77b374f5"},
]

def source_url(src):
    if src["kind"] == "github":
        return ("https://raw.githubusercontent.com/" + src["repo"] + "/" +
                src["rev"] + "/" + urllib.parse.quote(src["path"], safe="/"))
    if src["kind"] == "hf":
        return ("https://huggingface.co/datasets/Dizzzy0x00/"
                "LLMGen-Phishing-Email-Dataset/resolve/" + src["rev"] +
                "/" + urllib.parse.quote(src["path"]) + "?download=true")
    return ("https://zenodo.org/records/20250116/files/" +
            urllib.parse.quote(src["path"]) + "?download=1")

def sha_and_md5(path):
    sh, md5 = hashlib.sha256(), hashlib.md5()  # MD5 is solely a published-file identity check
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024 * 1024),b""):
            sh.update(block)
            md5.update(block)
    return sh.hexdigest(), md5.hexdigest()

def git_blob(path):
    path = Path(path)
    sh = hashlib.sha1(b"blob " + str(path.stat().st_size).encode() + b"\x00")
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024 * 1024),b""):
            sh.update(block)
    return sh.hexdigest()

def verify_source(path, src):
    path = Path(path)
    if not path.is_file():
        return None
    size = path.stat().st_size
    if size <= 0 or size > src.get("size_cap", src.get("size", 0) + 1024):
        return None
    if src["kind"] == "github" and (size != src["size"] or git_blob(path) != src["blob"]):
        return None
    sha, md5 = sha_and_md5(path)
    if src["kind"] == "zenodo" and md5 != src["md5"]:
        return None
    return {"sha256":sha,"md5":md5,"bytes":size}

def acquire(src):
    # A HF file without a published/pinned content digest is downloaded directly
    # from the frozen revision; an arbitrary attached file is not treated as verified.
    if src["kind"] != "hf":
        input_root = Path("/kaggle/input")
        if input_root.is_dir():
            for path in sorted(input_root.rglob(src["name"])):
                match = verify_source(path,src)
                if match is not None:
                    return path,match,"attached_pinned_input"
    dst = TEMP / src["name"]
    if src["kind"] != "hf":
        match = verify_source(dst,src)
        if match is not None:
            return dst,match,"verified_temporary_cache"
    tmp = TEMP / (src["name"]+".partial")
    size_cap = src.get("size_cap", src.get("size",0)+1024)
    started = time.monotonic()
    try:
        req = urllib.request.Request(source_url(src),headers={
            "User-Agent":"AcademicDatasetIntegrityAudit/1.0", "Accept":"application/octet-stream"})
        with urllib.request.urlopen(req,timeout=180) as resp:
            declared = resp.headers.get("Content-Length")
            if declared and int(declared)>size_cap:
                raise ValueError("remote_size_cap_exceeded")
            nbytes=0
            with tmp.open("wb") as f:
                while True:
                    chunk=resp.read(1024*1024)
                    if not chunk: break
                    nbytes+=len(chunk)
                    if nbytes>size_cap:
                        raise ValueError("download_size_cap_exceeded")
                    f.write(chunk)
        check=verify_source(tmp,src)
        if check is None:
            raise ValueError("download_integrity_mismatch")
        tmp.replace(dst)
        return dst,check,"official_revision_download"
    finally:
        tmp.unlink(missing_ok=True)

def normalize_key(name):
    return re.sub(r"[^a-z0-9]","",str(name).casefold())

def safe_columns(row):
    return sorted([k if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_ ]{0,50}",str(k)) else "<unusual>"
                   for k in row.keys()])[:100]

def short_aggregate_value(x):
    if isinstance(x, bool): return str(x).lower()
    if x is None:return "<missing>"
    if isinstance(x,(int,float)) and len(str(x))<15:return str(x)
    if isinstance(x,str):
        y=x.strip()
        # Only conventional semantic category literals; never emit arbitrary email text.
        if normalize_key(y) in {
            "phishing","spam","valid","legitimate","ham","benign","normal",
            "human","humanwritten","ai","llm","aigenerated","synthetic",
            "0","1","2","3","true","false","gpt4o","deepseek","manual",
            "randomapi","original","rephrased","unknown","notavailable",
            "gpt","en","zh","english","chinese"}:
            return y[:32]
    return "<other_or_redacted>"

def _field(row, choices):
    norm={normalize_key(k):k for k in row}
    for c in choices:
        if normalize_key(c) in norm:
            return row[norm[normalize_key(c)]], norm[normalize_key(c)]
    return None,None

def text_key(row):
    subject,_=_field(row,["subject","email_subject"])
    body,_=_field(row,["body","email_body","content","email_text","text","message"])
    if not isinstance(subject,str):subject=""
    if not isinstance(body,str):body=""
    if not subject and not body:return None
    txt=" ".join((subject+"\n"+body).casefold().split())
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()

def records_json(path):
    with Path(path).open("r",encoding="utf-8-sig") as f:
        obj=json.load(f)
    if isinstance(obj,list):
        rows=obj
    elif isinstance(obj,dict):
        rows=None
        for k in ["emails","data","records","items","dataset","results"]:
            if isinstance(obj.get(k),list):
                rows=obj[k];break
        if rows is None and obj and all(isinstance(v,dict) for v in obj.values()):
            rows=list(obj.values())
        if rows is None:
            raise ValueError("unsupported_json_container")
    else:
        raise ValueError("unsupported_json_container")
    for row in rows:
        if not isinstance(row,dict):
            raise ValueError("non_object_json_record")
        yield row

def records_jsonl(path):
    with Path(path).open("r",encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                row=json.loads(line)
                if not isinstance(row,dict):raise ValueError("non_object_jsonl_record")
                yield row

def records_csv(path):
    with Path(path).open("r",encoding="utf-8-sig",newline="") as f:
        reader=csv.DictReader(f,strict=True)
        hdr=reader.fieldnames or []
        if not hdr or len(hdr)!=len(set(hdr)):
            raise ValueError("bad_csv_headers")
        for row in reader:
            if None in row or any(v is None for v in row.values()):
                raise ValueError("malformed_csv_record")
            yield row

def scan_rows(iterator):
    rows=0
    text_counts=Counter()
    labels=Counter()
    origins=Counter()
    parents=Counter()
    missing_parent=0
    schema=set()
    label_fields=Counter()
    text_present=0
    for row in iterator:
        rows+=1
        if rows==1: schema.update(safe_columns(row))
        label,lf=_field(row,["true_category","true_label","label","type","category","ground_truth"])
        if lf is not None:
            labels[short_aggregate_value(label)]+=1
            label_fields[lf]+=1
        origin,of=_field(row,["created_by","origin","human_or_ai","rephrase_source","generator"])
        if of is not None: origins[short_aggregate_value(origin)]+=1
        parent,pf=_field(row,["Original_ID","original_id","parent_id","source_id","original_email_id"])
        if pf is not None:
            if parent is None or (isinstance(parent,str) and not parent.strip()):
                missing_parent+=1
            elif isinstance(parent,(str,int)):
                # Values never leave memory; only group counts are exported.
                parents[str(parent)]+=1
            else:
                missing_parent+=1
        key=text_key(row)
        if key:
            text_present+=1
            text_counts[key]+=1
    if not rows:raise ValueError("zero_record_source")
    return {
        "rows":rows,"schema_columns":sorted(schema),
        "candidate_label_field_counts":dict(label_fields),
        "candidate_label_values":dict(labels),
        "origin_or_generator_field_values":dict(origins),
        "rows_with_subject_body_or_content":text_present,
        "normalized_extra_duplicate_text_rows":sum(n-1 for n in text_counts.values() if n>1),
        "normalized_unique_text_keys":len(text_counts),
        "parent_key_unique_count":len(parents),
        "rows_missing_parent_when_field_exists":missing_parent,
        "parent_group_size_histogram":dict(Counter(parents.values()).most_common(12)),
    },text_counts

def scan_zip(path):
    with zipfile.ZipFile(path) as z:
        entries=z.infolist()
        if len(entries)>2000:raise ValueError("too_many_zip_entries")
        if sum(i.file_size for i in entries)>350_000_000:raise ValueError("zip_expanded_size_cap")
        extensions=Counter()
        csv_tables=[]
        for i in entries:
            if i.is_dir():continue
            if i.file_size>100_000_000 or (i.compress_size>0 and i.file_size/i.compress_size>150):
                raise ValueError("zip_entry_size_or_ratio_exceeded")
            ext=Path(i.filename).suffix.lower() or "<none>"
            extensions[ext]+=1
            if ext==".csv" and i.file_size<=35_000_000:
                with z.open(i) as blob:
                    with io.TextIOWrapper(blob,encoding="utf-8-sig",newline="") as f:
                        reader=csv.DictReader(f,strict=True)
                        headers=reader.fieldnames or []
                        if not headers or len(headers)!=len(set(headers)):
                            raise ValueError("invalid_embedded_csv")
                        count=0
                        label_counts=Counter()
                        for row in reader:
                            if None in row or any(v is None for v in row.values()):
                                raise ValueError("malformed_embedded_csv")
                            count+=1
                            label,lf=_field(row,["true_category","true_label","label","type","category","ground_truth"])
                            if lf: label_counts[short_aggregate_value(label)]+=1
                        csv_tables.append({"rows":count,"columns":safe_columns(dict.fromkeys(headers)),
                                           "candidate_label_values":dict(label_counts)})
        return {"archive_entries":len(entries),"archive_extension_counts":dict(extensions),
                "embedded_csv_tables":csv_tables,
                "interpretation":"D5 is phishing-only origin-attribution, NOT primary phishing-vs-legitimate gold"}

def get_gpu_info():
    try:
        r=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free,compute_cap",
                          "--format=csv,noheader,nounits"],check=False,capture_output=True,text=True,timeout=6)
        return [x.strip() for x in r.stdout.splitlines() if x.strip()] if r.returncode==0 else []
    except (OSError,subprocess.TimeoutExpired):
        return []

def memory_info():
    output={}
    try:
        with open("/proc/meminfo",encoding="ascii") as f:
            for line in f:
                k,_,v=line.partition(":")
                if k in ("MemTotal","MemAvailable"):
                    output[k+"_KiB"]=int(v.strip().split()[0])
    except (OSError,ValueError):pass
    return output

def peak_rss_mb():
    try:
        import resource
        v=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return round(v/(1024 if platform.system()=="Linux" else 1024*1024),2)
    except (OSError,ImportError):return None

def environment():
    installed={}
    for p in ("numpy","pandas","torch","transformers","datasets","scikit-learn","scipy"):
        try:installed[p]=importlib.metadata.version(p)
        except importlib.metadata.PackageNotFoundError:installed[p]="not_installed"
    disk=shutil.disk_usage(TEMP)
    return {"utc":datetime.now(timezone.utc).isoformat(),"runtime":"Kaggle" if ON_KAGGLE else "local",
            "python":sys.version.split()[0],"system":platform.platform(),
            "cpu_logical_cores":os.cpu_count(),
            "memory":memory_info(),"gpu_observed":get_gpu_info(),"packages":installed,
            "scratch_disk_total_bytes":disk.total,"scratch_disk_free_bytes":disk.free,
            "source_revision":{"D2":REV_D2,"D3":REV_D3,"D4":REV_D4,
                               "D5":"Zenodo record 20250116 / pinned MD5"}}

def synthetic_test():
    with tempfile.TemporaryDirectory() as t:
        p=Path(t)/"synthetic.csv"
        p.write_text('subject,body,label,Original_ID\nA,hello,1,42\na,HELLO,1,42\nB,goodbye,0,43\n',encoding="utf-8")
        result,keys=scan_rows(records_csv(p))
        assert result["rows"]==3 and result["normalized_extra_duplicate_text_rows"]==1
        assert result["parent_key_unique_count"]==2 and set(result["candidate_label_values"])=={"0","1"}
        q=Path(t)/"synthetic.json";q.write_text('[{"Subject":"A","Body":"hello","Type":"Phishing"}]')
        parsed,_=scan_rows(records_json(q))
        assert parsed["candidate_label_values"]["Phishing"]==1
        with zipfile.ZipFile(Path(t)/"synthetic.zip","w") as z:
            z.writestr("folder/data.csv","label,feature\n1,2\n0,4\n")
        zrep=scan_zip(Path(t)/"synthetic.zip")
        assert zrep["embedded_csv_tables"][0]["rows"]==2
    return "PASS"

def main():
    start=time.perf_counter()
    test=synthetic_test()
    env=environment()
    (OUT/"environment.json").write_text(json.dumps(env,indent=2)+"\n",encoding="utf-8")
    results={"release":RELEASE,"time_utc":datetime.now(timezone.utc).isoformat(),
             "synthetic_self_test":test,"files":{},"source_family_status":{},
             "research_gate":"DESCRIPTIVE_INTAKE_ONLY",
             "primary_training_gate":"NO_GO",
             "rights_and_label_verification":"PENDING",
             "notes":["Source checksum does not verify class semantics or upstream data rights.",
                      "D2 rephrase pipeline names are NOT verified per-record LLM identities.",
                      "D3 Original_ID is a candidate relation; security-changing edits are NOT assumed label-preserving.",
                      "D4 English vs Chinese language is confounded with stated generator.",
                      "D5 origin attribution is NOT phishing-vs-legitimate classification.",
                      "No raw emails, personal information, URLs or per-row text hashes exported."]}
    timings={}
    tmp_disk_before=shutil.disk_usage(TEMP).free
    for src in SOURCES:
        t=time.perf_counter()
        entry={"source_id":src["id"],"family":src["family"],
               "source_revision":src.get("rev","Zenodo 20250116"),
               "source_url":source_url(src),"status":"INCOMPLETE"}
        try:
            if shutil.disk_usage(TEMP).free < max(220_000_000,3*src.get("size_cap",src.get("size",0))):
                raise ValueError("insufficient_temporary_disk")
            if src["kind"]=="github":
                available=env["memory"].get("MemAvailable_KiB",0)*1024
                if available and src["size"]*12 > available*0.6:
                    raise ValueError("insufficient_memory_for_json_parse")
            file,hashes,mode=acquire(src)
            entry.update({"source_sha256":hashes["sha256"],"bytes":hashes["bytes"],
                          "source_mode":mode,"checksum_verification":
                          "GIT_BLOB_PINNED" if src["kind"]=="github" else
                          "ZENODO_MD5_PINNED" if src["kind"]=="zenodo" else
                          "PINNED_REVISION_ONLY_SHA256_RECORDED"})
            if file.suffix==".zip":
                entry.update(scan_zip(file))
            elif file.suffix==".json":
                info,_=scan_rows(records_json(file));entry.update(info)
            elif file.suffix==".jsonl":
                info,_=scan_rows(records_jsonl(file));entry.update(info)
            elif file.suffix==".csv":
                info,_=scan_rows(records_csv(file));entry.update(info)
            else:raise ValueError("unsupported_source_format")
            entry["status"]="PASS_DESCRIPTIVE"
            print(src["id"],": PASS, rows",entry.get("rows","ZIP metadata"),
                  ", elapsed_s",round(time.perf_counter()-t,1))
        except (OSError,ValueError,UnicodeError,csv.Error,zipfile.BadZipFile,
                json.JSONDecodeError,TimeoutError,MemoryError) as e:
            entry["status"]="FAILED_OR_UNAVAILABLE"
            entry["error_type"]=type(e).__name__
            print(src["id"],": FAILED",type(e).__name__)
        timings[src["id"]]=round(time.perf_counter()-t,3)
        results["files"][src["id"]]=entry
    for fam in ("D2","D3","D4","D5"):
        items=[x for x in results["files"].values() if x["family"]==fam]
        results["source_family_status"][fam]={
            "passed":sum(x["status"]=="PASS_DESCRIPTIVE" for x in items),
            "expected":len(items),
            "status":"DESCRIPTIVE_PASS" if items and all(x["status"]=="PASS_DESCRIPTIVE" for x in items)
                     else "INCOMPLETE"}
    overall=all(x["status"]=="PASS_DESCRIPTIVE" for x in results["files"].values())
    results["status"]="ALL_SOURCE_AUDITS_COMPLETED_RESEARCH_GATES_PENDING" if overall else "PARTIAL_SOURCE_AUDIT"
    (OUT/"d2_d5_aggregate_report.json").write_text(
        json.dumps(results,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
    resources={"total_elapsed_seconds":round(time.perf_counter()-start,3),
               "per_source_elapsed_seconds":timings,
               "peak_process_rss_mb":peak_rss_mb(),
               "scratch_disk_free_before_bytes":tmp_disk_before,
               "scratch_disk_free_after_bytes":shutil.disk_usage(TEMP).free,
               "gpu_training_performed":False,
               "gpu_hours_used_by_training":0,
               "gpu_runtime_metrics":"not_applicable_cpu_only",
               "data_download_total_bytes":sum(x.get("bytes",0) for x in results["files"].values()
                                                if x.get("source_mode")=="official_revision_download")}
    (OUT/"resource_usage.json").write_text(json.dumps(resources,indent=2)+"\n",encoding="utf-8")
    manifest={"release":RELEASE,"report_status":results["status"],"python":env["python"],
              "source_revision":env["source_revision"],"input_sha256_by_source":
              {k:v.get("source_sha256") for k,v in results["files"].items()},
              "generated_artifacts":["environment.json","resource_usage.json",
                                     "d2_d5_aggregate_report.json","run_manifest.json","summary.txt"],
              "reproducibility_limitation":"Notebook 02 GitHub commit must be recorded externally; run code must be versioned."}
    (OUT/"run_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
    summary=["Notebook 02 D2–D5","Status: "+results["status"],
             "Synthetic self-test: "+test,
             "Python: "+env["python"],"CPU cores: "+str(env["cpu_logical_cores"]),
             "GPU: "+str(len(env["gpu_observed"]))+" observed",
             "Peak process RSS MB: "+str(resources["peak_process_rss_mb"]),
             "Total elapsed seconds: "+str(resources["total_elapsed_seconds"]),
             "Source audits:"]
    summary += [f"  {k}: {v['passed']}/{v['expected']} {v['status']}"
                for k,v in results["source_family_status"].items()]
    summary += ["Primary phishing detector TRAINING: NO-GO",
                "Rights, label semantics, upstream provenance, parent validity: PENDING",
                "Send only aggregate report/environment/resource_usage/summary files.",
                "Do not publish raw email source files from /kaggle/temp or attached private inputs."]
    (OUT/"summary.txt").write_text("\n".join(summary)+"\n",encoding="utf-8")
    print("\n"+"\n".join(summary))
    print("Shareable output directory:",OUT)
    return results

N02_RESULT = main()

## What to send back

After **Run All**, attach only the files in `/kaggle/working/phishing_d2_d5_evidence/` (especially `d2_d5_aggregate_report.json`, `environment.json`, `resource_usage.json`, and `summary.txt`). Do not share raw source emails, generated private index files, or run source scripts. A source that fails availability/format/integrity checks is reported as FAILED, never silently replaced. The next notebook will use this audit evidence to determine what pair data can actually be verified.